# Part 1: Expectation-Maximization (EM) Algorithm

## Objective

The objective of this notebook is to implement the Expectation-Maximization (EM) algorithm from scratch to separate a mixed dataset of Father and Child heights into two Gaussian distributions without using their labels.

The implementation follows the mathematical steps of the EM algorithm:

- Load and mix the dataset
- Initialize model parameters
- Compute Gaussian probabilities
- Perform the Expectation (E-Step)
- Perform the Maximization (M-Step)
- Calculate Log-Likelihood
- Repeat until convergence
- Predict the probability that a random height belongs to a Father or a Child

## Dataset

The Galton Families dataset contains measurements of family members' heights.

For this assignment, we selected:

- Father Heights
- Child Heights

The two groups are combined into a single dataset and shuffled so that the original labels are hidden. This simulates a real-world clustering problem where class labels are unknown.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#==================
# Load dataset
#==================

df = pd.read_csv('data/GaltonFamilies.csv')
print(df.head())

In [ ]:
#==================
# Mix dataset (For having a new dataset with no structure relation with the original one)
#==================

father = df['father'].to_numpy()
children = df['childHeight'].to_numpy()

heights = np.concatenate((father, children))

np.random.seed(42)
np.random.shuffle(heights)

print(heights[:10])

In [ ]:
#==================
# Visualize data
#==================

plt.hist(heights, bins=30)
plt.xlabel('Height')
plt.ylabel('Frequency')
plt.title('Mixed heights')
plt.show()

In [ ]:
#==========================
# Initialize parameters (Guess initial parameters for the EM algorithm / Mean / Variance / Mixing coefficients)
#==========================

mu1 = np.percentile(heights, 25)
mu2 = np.percentile(heights, 75)

sigma1 = np.std(heights)
sigma2 = np.std(heights)

pi1 = 0.5
pi2 = 0.5

print(mu1)
print(mu2)
print(sigma1)
print(sigma2)
print(pi1)
print(pi2)

## Gaussian (Normal) Distribution

The EM algorithm assumes that each population follows a Gaussian distribution.

The Gaussian Probability Density Function (PDF) is

\[
f(x)=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
\]

where

- μ = Mean
- σ = Standard Deviation

The PDF computes how likely a particular height belongs to a Gaussian distribution.

In [ ]:
#==========================
# Build Gaussian function (Probability Density Function Equation)
#==========================

def gaussian(data, mu, sigma):
    coefficient = 1 / (np.sqrt(2 * np.pi) * sigma)
    exponent = np.exp(-((data - mu) ** 2) / (2 * sigma ** 2))
    return coefficient * exponent

## Expectation Step (E-Step)

During the Expectation Step, the algorithm computes the posterior probability (responsibility) that every height belongs to each Gaussian distribution.

For every observation, we calculate

- Probability of belonging to Gaussian 1
- Probability of belonging to Gaussian 2

The responsibilities always sum to 1.

These probabilities are later used to update the model parameters.

In [ ]:
#==========================
# Implement E-step (For finding the responsibilities for in the dataset which means probabilities of each data point belonging to each Gaussian distribution)
#==========================

def expectation_step(data, mu1, mu2, sigma1, sigma2, pi1, pi2):

    # Probability under Gaussian 1 (Children)
    prob1 = pi1 * gaussian(data, mu1, sigma1)

    # Probability under Gaussian 2 (Fathers)
    prob2 = pi2 * gaussian(data, mu2, sigma2)

    # Total probability
    total = prob1 + prob2

    # Responsibilities
    r1 = prob1 / total
    r2 = prob2 / total

    return r1, r2

## Maximization Step (M-Step)

During the Maximization Step, the parameters of each Gaussian distribution are updated using the responsibilities computed during the E-Step.

The updated parameters include

- Mean (μ)
- Standard Deviation (σ)
- Mixing Coefficient (π)

These updated values better describe the two hidden populations.

In [ ]:
#==========================
# Implement M-step (For updating the parameters of the Gaussian distributions based on the responsibilities calculated in the E-step)
#==========================

def maximization_step(data, r1, r2):

    mu1 = np.sum(r1 * data) / np.sum(r1)
    mu2 = np.sum(r2 * data) / np.sum(r2)

    sigma1 = np.sqrt(
        np.sum(r1 * (data - mu1) ** 2) / np.sum(r1)
    )

    sigma2 = np.sqrt(
        np.sum(r2 * (data - mu2) ** 2) / np.sum(r2)
    )

    pi1 = np.mean(r1)
    pi2 = np.mean(r2)

    return mu1, mu2, sigma1, sigma2, pi1, pi2

## Log-Likelihood

The Log-Likelihood measures how well the current Gaussian distributions explain the observed data.

After every iteration, the Log-Likelihood should increase (or become less negative).

When the Log-Likelihood no longer changes significantly, the EM algorithm has converged.

In [ ]:
#==========================
# Compute log-likelihood (For evaluating whether model parameters are improving or not / The less the more likely the model is to fit the data)
#==========================

def log_likelihood(data, mu1, mu2, sigma1, sigma2, pi1, pi2):
    
    likelihood = (
        pi1 * gaussian(data, mu1, sigma1)
        +
        pi2 * gaussian(data, mu2, sigma2)
    )
    
    likelihood = np.clip(likelihood, 1e-12, None)

    return np.sum(np.log(likelihood))

In [ ]:
#==========================
# Run EM loop (For iteratively performing the E-step and M-step until convergence)
#==========================

max_iterations = 100
tolerance = 1e-6

previous_ll = -np.inf

print(
    f"{'Iter':<5}"
    f"{'μ1':>10}"
    f"{'μ2':>10}"
    f"{'σ1²':>10}"
    f"{'σ2²':>10}"
    f"{'π1':>10}"
    f"{'π2':>10}"
    f"{'LogLik':>15}"
)

for iteration in range(max_iterations):

    # E-Step
    r1, r2 = expectation_step(
        heights,
        mu1,
        mu2,
        sigma1,
        sigma2,
        pi1,
        pi2
    )

    # M-Step
    mu1, mu2, sigma1, sigma2, pi1, pi2 = maximization_step(
        heights,
        r1,
        r2
    )

    # Log Likelihood
    ll = log_likelihood(
        heights,
        mu1,
        mu2,
        sigma1,
        sigma2,
        pi1,
        pi2
    )

    # Check for convergence
    if abs(ll - previous_ll) < tolerance:
        print(f"Converged after {iteration} iterations.")
        break

    previous_ll = ll

    print(
        f"{iteration:<5}"
        f"{mu1:>10.2f}"
        f"{mu2:>10.2f}"
        f"{sigma1**2:>10.2f}"
        f"{sigma2**2:>10.2f}"
        f"{pi1:>10.3f}"
        f"{pi2:>10.3f}"
        f"{ll:>15.2f}"
    )

## Height Classification

After training, the model can classify any new height.

Instead of making a hard decision, the model computes

- Probability the height belongs to a Child
- Probability the height belongs to a Father

The class with the highest posterior probability becomes the prediction.

In [ ]:
#==========================
# Random heights (For handling random height values during presentation)
#==========================

height = float(input("Enter a height: "))

def classify_height(height, mu1, mu2, sigma1, sigma2, pi1, pi2):

    p1 = pi1 * gaussian(height, mu1, sigma1)
    p2 = pi2 * gaussian(height, mu2, sigma2)

    total = p1 + p2

    child_probability = p1 / total
    father_probability = p2 / total

    print(f"Height: {height:.2f}")
    print(f"Probability Child : {child_probability:.4f}")
    print(f"Probability Father: {father_probability:.4f}")

    if child_probability > father_probability:
        print("Prediction: Child")
    else:
        print("Prediction: Father")

classify_height(
    height,
    mu1,
    mu2,
    sigma1,
    sigma2,
    pi1,
    pi2
)

In [ ]:
# ==========================
# Plot the final Gaussian distributions after EM convergence
# ==========================

x = np.linspace(min(heights), max(heights), 500)

plt.hist(heights, bins=30, density=True, alpha=0.6)

plt.plot(x, pi1 * gaussian(x, mu1, sigma1), label="Children")
plt.plot(x, pi2 * gaussian(x, mu2, sigma2), label="Fathers")

plt.legend()
plt.show()

## Why Should We Not Split the Dataset at the Global Mean?

Simply splitting the dataset at the global mean assumes that every observation below the mean belongs to one group and every observation above the mean belongs to the other.

This approach ignores the fact that the two Gaussian distributions overlap.

Many heights near the center could reasonably belong to either Fathers or Children.

The Expectation-Maximization algorithm instead computes posterior probabilities for every observation, allowing uncertain observations to partially belong to both groups during training.

This probabilistic approach produces a much more accurate model than using a fixed threshold at the global mean.